# Eksperimen 8: LGBM (Early Stopping) + ExtraTrees Stacking
**Strategi: Anti-Overfitting Ensemble dengan Diversitas Model Tinggi**

Berdasarkan diagnosa Eksperimen 7, XGBoost dan CatBoost tidak berkontribusi positif (bobot negatif/nol). Eksperimen ini menggantikan mereka dengan:
1. **LightGBM + Early Stopping**: Anti-overfitting otomatis di setiap K-Fold. Model berhenti menambah pohon begitu performa validasi tidak membaik.
2. **ExtraTreesRegressor**: Model pohon yang sangat berbeda secara fundamental dari LGBM. ExtraTrees memilih titik split secara acak, sehingga korelasinya rendah terhadap LGBM dan memberikan diversitas prediksi yang sesungguhnya untuk Ridge Stacking.

In [1]:
import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
from sklearn.model_selection import TimeSeriesSplit
import optuna
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 150)
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Pemuatan Data

In [2]:
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')
env_data = pd.read_csv('../data/raw/data_pendukung/data_lingkungan.csv')
coords = pd.read_csv('../data/raw/data_pendukung/koordinat_pos.csv')

train['datetime'] = pd.to_datetime(train['datetime'])
test['datetime'] = pd.to_datetime(test['id'].str[:19])
test['nama_pos'] = test['id'].str[22:]
env_data['datetime'] = pd.to_datetime(env_data['datetime'])

## 2. EDA & CEDA

In [3]:
print("=== Dimensi Matriks ===")
print("Train:", train.shape)
print("Test:", test.shape)
print("Lingkungan:", env_data.shape)

print("\n=== Missing Values Data Lingkungan ===")
print(env_data.isnull().sum()[env_data.isnull().sum() > 0])

print("\n=== Batas Kronologis ===")
print("Akhir Train:", train['datetime'].max())
print("Awal Test  :", test['datetime'].min())

=== Dimensi Matriks ===
Train: (84396, 3)
Test: (21780, 3)
Lingkungan: (888480, 27)

=== Missing Values Data Lingkungan ===
soil_moisture_0_7cm          720
soil_moisture_7_28cm         720
soil_moisture_28_100cm       720
soil_moisture_100_255cm      720
surface_pressure_hpa         720
pressure_msl_hpa             720
rmm1                         720
rmm2                         720
mjo_phase                    720
mjo_amplitude                720
mjo_active                   720
nino_34                    12960
dtype: int64

=== Batas Kronologis ===
Akhir Train: 2025-09-18 18:00:00
Awal Test  : 2025-09-19 06:00:00


### 2.1 Distribusi TMA per Pos & Last Known Condition

In [4]:
station_stats = train.groupby('nama_pos')['tma_mdpl'].agg(['mean', 'std', 'min', 'max'])
print("=== Distribusi TMA per Pos ===")
print(station_stats.sort_values('mean', ascending=False).round(2).to_string())

last_known = (
    train.sort_values('datetime')
    .groupby('nama_pos')
    .agg(tma_last_known=('tma_mdpl', 'last'))
    .reset_index()
)

=== Distribusi TMA per Pos ===
                             mean   std     min     max
nama_pos                                               
Ngadipiro                  143.56  0.33  143.18  146.31
Ngrembang                  140.05  0.29  139.85  143.79
Wonogiri Dam               132.34  3.30  125.54  137.36
Badegan                    122.40  0.31  121.90  123.96
Colo Weir                  107.79  1.12  102.19  109.70
Kali Pepe - Tugu Boto       94.82  0.45   94.35  100.42
Peren                       91.31  2.02   90.11  170.10
Jarum                       90.72  3.92   89.29  250.14
Sekayu                      87.27  0.66   86.61   92.32
Kali Anyar - Kreteg Abang   86.46  4.46   84.44  323.21
Serenan                     86.32  0.70   85.47   90.98
Kali Pepe - PTPN            82.55  2.00    0.00  138.07
Jurug                       78.82  1.94    0.00   86.46
Kedungupit                  64.87  3.08   62.13  207.74
Kajangan                    51.04  2.61   49.18  172.77
Ketonggo         

### 2.2 CEDA: Korelasi Variabel Lingkungan

In [5]:
train_temp = pd.merge(train, env_data, on=['datetime', 'nama_pos'], how='left')
num_cols = train_temp.select_dtypes(include=[np.number]).columns
corr = train_temp[num_cols].corr()['tma_mdpl'].sort_values(ascending=False)
print("=== Top Korelasi terhadap TMA ===")
print(corr.head(6).round(4))
print("...")
print(corr.tail(5).round(4))

=== Top Korelasi terhadap TMA ===
tma_mdpl                   1.0000
soil_moisture_100_255cm    0.1894
built_surface_m2           0.1793
soil_moisture_28_100cm     0.1270
soil_moisture_7_28cm       0.1238
soil_moisture_0_7cm        0.1058
Name: tma_mdpl, dtype: float64
...
rainfall_max_24h_mm    -0.0251
temperature_c          -0.0770
dew_point_c            -0.1217
landcover_class        -0.1573
surface_pressure_hpa   -0.9474
Name: tma_mdpl, dtype: float64


## 3. Station Profile (Anti-Leakage)
Statistik per pos dihitung **murni dari train**. Last Known TMA digunakan sebagai sinyal awal kondisi sungai.

In [6]:
station_profile = train.groupby('nama_pos')['tma_mdpl'].agg(
    tma_mean='mean',
    tma_std='std',
    tma_p25=lambda x: x.quantile(0.25),
    tma_p75=lambda x: x.quantile(0.75)
).reset_index()
station_profile['tma_std'] = station_profile['tma_std'].fillna(1.0)

global_mean = train['tma_mdpl'].mean()
global_std = train['tma_mdpl'].std()

station_profile = pd.merge(station_profile, last_known, on='nama_pos', how='left')
station_profile['tma_last_known_norm'] = (
    (station_profile['tma_last_known'] - station_profile['tma_mean']) / station_profile['tma_std']
)

## 4. Preprocessing Adaptif

In [7]:
env_data = env_data.sort_values(['nama_pos', 'datetime'])

macro_cols = ['nino_34', 'mjo_phase', 'mjo_amplitude', 'mjo_active', 'rmm1', 'rmm2']
dynamic_cols = ['surface_pressure_hpa', 'pressure_msl_hpa', 'soil_moisture_0_7cm',
                'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm']

for c in macro_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].ffill().bfill()

for c in dynamic_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].apply(
        lambda x: x.interpolate(method='linear').bfill().ffill()
    ).reset_index(level=0, drop=True)

### 4.1 Agregasi & Penggabungan Spasial

In [8]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
coords['spatial_cluster'] = kmeans.fit_predict(coords[['latitude', 'longitude']])

def aggregate_env_data(df):
    agg_funcs = {col: 'mean' for col in df.columns if col not in ['nama_pos', 'landcover_name', 'datetime']}
    agg_funcs['rainfall_mm'] = 'sum'
    agg_funcs['rainfall_openmeteo_mm'] = 'sum'
    agg_funcs['rainfall_max_24h_mm'] = 'max'
    df_indexed = df.set_index('datetime')
    return df_indexed.groupby(['nama_pos', pd.Grouper(freq='3h', label='right', closed='right')]).agg(agg_funcs).reset_index()

env_agg = aggregate_env_data(env_data)

test['tma_mdpl'] = np.nan
all_data = pd.concat([train, test], ignore_index=True)
all_data = all_data.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

all_data = pd.merge(all_data, env_agg, on=['datetime', 'nama_pos'], how='left')
all_data = pd.merge(all_data, coords, on='nama_pos', how='left')
all_data = pd.merge(all_data, station_profile, on='nama_pos', how='left')

all_data['tma_mean'] = all_data['tma_mean'].fillna(global_mean)
all_data['tma_std'] = all_data['tma_std'].fillna(global_std)
all_data['tma_last_known'] = all_data['tma_last_known'].fillna(global_mean)
all_data['tma_last_known_norm'] = all_data['tma_last_known_norm'].fillna(0)

## 5. Deep Feature Engineering

In [9]:
le = LabelEncoder()
all_data['nama_pos_encoded'] = le.fit_transform(all_data['nama_pos'])

all_data['month'] = all_data['datetime'].dt.month
all_data['hour'] = all_data['datetime'].dt.hour
all_data['day_of_year'] = all_data['datetime'].dt.dayofyear
all_data['sin_hour'] = np.sin(2 * np.pi * all_data['hour'] / 24)
all_data['cos_hour'] = np.cos(2 * np.pi * all_data['hour'] / 24)
all_data['sin_month'] = np.sin(2 * np.pi * all_data['month'] / 12)
all_data['cos_month'] = np.cos(2 * np.pi * all_data['month'] / 12)

all_data['runoff_factor'] = all_data['rainfall_mm'] * all_data['soil_moisture_0_7cm']
all_data['pressure_drop'] = all_data.groupby('nama_pos')['surface_pressure_hpa'].diff(1).fillna(0)
all_data['rainfall_intensity'] = all_data['rainfall_mm'] / (all_data['soil_moisture_0_7cm'] + 1e-6)
all_data['temp_humidity_index'] = all_data['temperature_c'] * all_data['humidity_pct'] / 100

windows = [4, 8, 24, 56]
for w in windows:
    all_data[f'rainfall_roll_{w}'] = all_data.groupby('nama_pos')['rainfall_mm'].transform(
        lambda x: x.rolling(window=w, min_periods=1).sum()
    )
    all_data[f'soil_roll_{w}'] = all_data.groupby('nama_pos')['soil_moisture_0_7cm'].transform(
        lambda x: x.rolling(window=w, min_periods=1).mean()
    )
    all_data[f'pressure_roll_{w}'] = all_data.groupby('nama_pos')['surface_pressure_hpa'].transform(
        lambda x: x.rolling(window=w, min_periods=1).mean()
    )
    all_data[f'temp_roll_{w}'] = all_data.groupby('nama_pos')['temperature_c'].transform(
        lambda x: x.rolling(window=w, min_periods=1).mean()
    )

## 6. Target Normalization per Pos

In [10]:
train_mask = all_data['tma_mdpl'].notnull()

all_data['tma_normalized'] = np.nan
all_data.loc[train_mask, 'tma_normalized'] = (
    (all_data.loc[train_mask, 'tma_mdpl'] - all_data.loc[train_mask, 'tma_mean']) /
    all_data.loc[train_mask, 'tma_std']
)

print("Distribusi target setelah normalisasi:")
print(all_data.loc[train_mask, 'tma_normalized'].describe().round(4))

Distribusi target setelah normalisasi:
count    84396.0000
mean         0.0000
std          0.9998
min        -41.2179
25%         -0.4486
50%         -0.1218
75%          0.2773
max         53.0309
Name: tma_normalized, dtype: float64


## 7. Training Setup

In [11]:
train_data = all_data[train_mask].sort_values('datetime').reset_index(drop=True)
test_data = all_data[~train_mask].sort_values('datetime').reset_index(drop=True)

drop_cols = ['datetime', 'nama_pos', 'tma_mdpl', 'tma_normalized', 'id', 'landcover_name']
features = [c for c in train_data.columns if c not in drop_cols]
target = 'tma_normalized'

X_full = train_data[features]
y_full = train_data[target]
X_test = test_data[features]

tscv = TimeSeriesSplit(n_splits=5)
print(f"Total fitur digunakan: {len(features)}")
print(features)

Total fitur digunakan: 61
['rainfall_mm', 'humidity_pct', 'wind_direction_deg', 'dew_point_c', 'cloud_cover_pct', 'temperature_c', 'wind_speed_kmh', 'rainfall_openmeteo_mm', 'rainfall_max_24h_mm', 'solar_radiation_mj_m2', 'soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm', 'surface_pressure_hpa', 'pressure_msl_hpa', 'built_surface_m2', 'landcover_class', 'rmm1', 'rmm2', 'mjo_phase', 'mjo_amplitude', 'mjo_active', 'nino_34', 'latitude', 'longitude', 'spatial_cluster', 'tma_mean', 'tma_std', 'tma_p25', 'tma_p75', 'tma_last_known', 'tma_last_known_norm', 'nama_pos_encoded', 'month', 'hour', 'day_of_year', 'sin_hour', 'cos_hour', 'sin_month', 'cos_month', 'runoff_factor', 'pressure_drop', 'rainfall_intensity', 'temp_humidity_index', 'rainfall_roll_4', 'soil_roll_4', 'pressure_roll_4', 'temp_roll_4', 'rainfall_roll_8', 'soil_roll_8', 'pressure_roll_8', 'temp_roll_8', 'rainfall_roll_24', 'soil_roll_24', 'pressure_roll_24', 'temp_roll_24', 'rain

### 7.1 Optuna Tuning: LightGBM (Anti-Overfitting Search Space)
Parameter pencarian dirancang untuk mencegah overfitting: `min_child_samples` dan regularisasi (`reg_alpha`, `reg_lambda`) diberikan ruang eksplorasi yang lebih agresif. Early Stopping diterapkan di setiap fold evaluasi Optuna.

In [12]:
def objective_lgb(trial):
    params = {
        'n_estimators': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 127),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'min_child_samples': trial.suggest_int('min_child_samples', 50, 300),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        'subsample_freq': 1,
        'random_state': 42,
        'verbose': -1
    }
    scores = []
    for train_idx, val_idx in tscv.split(X_full):
        X_tr, X_va = X_full.iloc[train_idx], X_full.iloc[val_idx]
        y_tr, y_va = y_full.iloc[train_idx], y_full.iloc[val_idx]
        m = lgb.LGBMRegressor(**params)
        m.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )
        scores.append(mean_squared_error(y_va, m.predict(X_va)))
    return np.mean(scores)

print("Memulai Optuna Tuning LightGBM dengan Early Stopping (60 trials)...")
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=60)
best_lgb = study_lgb.best_params
best_lgb.update({'n_estimators': 2000, 'random_state': 42, 'verbose': -1, 'subsample_freq': 1})
print(f"Best LightGBM Params: {best_lgb}")
print(f"Best CV MSE (normalized): {study_lgb.best_value:.6f}")

Memulai Optuna Tuning LightGBM dengan Early Stopping (60 trials)...
Best LightGBM Params: {'learning_rate': 0.04666728997587019, 'num_leaves': 69, 'max_depth': 6, 'subsample': 0.6744398076655964, 'colsample_bytree': 0.5036042481287949, 'min_child_samples': 246, 'reg_alpha': 1.7665418741015544, 'reg_lambda': 1.9684997964039777, 'n_estimators': 2000, 'random_state': 42, 'verbose': -1, 'subsample_freq': 1}
Best CV MSE (normalized): 0.669142


### 7.2 ExtraTreesRegressor Setup
ExtraTrees adalah *randomized* ensemble yang memilih split secara acak — sangat berbeda dari LGBM yang bersifat *boosted*. Keacakannya membuat model ini cenderung tidak overfitting dan memberikan diversitas prediksi yang tinggi untuk Ridge Stacking.

In [13]:
et_params = {
    'n_estimators': 500,
    'max_depth': 20,
    'min_samples_split': 10,
    'min_samples_leaf': 5,
    'max_features': 0.7,
    'random_state': 42,
    'n_jobs': -1
}

print("ExtraTreesRegressor akan dilatih dengan parameter:")
for k, v in et_params.items():
    print(f"  {k}: {v}")

ExtraTreesRegressor akan dilatih dengan parameter:
  n_estimators: 500
  max_depth: 20
  min_samples_split: 10
  min_samples_leaf: 5
  max_features: 0.7
  random_state: 42
  n_jobs: -1


## 8. K-Fold OOF Collection: LGBM + ExtraTrees
Melatih kedua model melintasi 5 lipatan waktu. Prediksi Out-of-Fold dikumpulkan untuk Ridge Meta-Learner. Early Stopping aktif pada setiap fold LGBM untuk mencegah overfitting secara real-time.

In [14]:
oof_lgb = np.zeros(len(X_full))
oof_et = np.zeros(len(X_full))

test_preds_lgb = np.zeros(len(X_test))
test_preds_et = np.zeros(len(X_test))

cv_fold_rmse = []

print("Memulai K-Fold OOF Collection (LGBM + ExtraTrees)...")
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_full)):
    X_tr, X_va = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_tr, y_va = y_full.iloc[train_idx], y_full.iloc[val_idx]

    val_mean = train_data.iloc[val_idx]['tma_mean'].values
    val_std = train_data.iloc[val_idx]['tma_std'].values

    m_lgb = lgb.LGBMRegressor(**best_lgb)
    m_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    m_et = ExtraTreesRegressor(**et_params)
    m_et.fit(X_tr, y_tr)

    oof_lgb[val_idx] = m_lgb.predict(X_va)
    oof_et[val_idx] = m_et.predict(X_va)

    blend_norm = (oof_lgb[val_idx] * 0.6) + (oof_et[val_idx] * 0.4)
    blend_abs = (blend_norm * val_std) + val_mean
    y_va_abs = (y_va.values * val_std) + val_mean
    fold_rmse = np.sqrt(mean_squared_error(y_va_abs, blend_abs))
    cv_fold_rmse.append(fold_rmse)
    print(f"Fold {fold+1} RMSE (abs): {fold_rmse:.4f} | Trees LGBM: {m_lgb.best_iteration_}")

    test_preds_lgb += m_lgb.predict(X_test) / tscv.n_splits
    test_preds_et += m_et.predict(X_test) / tscv.n_splits

print(f"\nRata-rata Naive Blend RMSE: {np.mean(cv_fold_rmse):.4f}")

Memulai K-Fold OOF Collection (LGBM + ExtraTrees)...
Fold 1 RMSE (abs): 3.8036 | Trees LGBM: 88
Fold 2 RMSE (abs): 1.8235 | Trees LGBM: 22
Fold 3 RMSE (abs): 0.6995 | Trees LGBM: 48
Fold 4 RMSE (abs): 1.3255 | Trees LGBM: 219
Fold 5 RMSE (abs): 1.0394 | Trees LGBM: 48

Rata-rata Naive Blend RMSE: 1.7383


## 9. Ridge Meta-Learner
Ridge menentukan bobot kontribusi LGBM dan ExtraTrees secara empiris berdasarkan OOF. Bobot optimal ditemukan dari data, bukan asumsi manual.

In [15]:
oof_matrix = np.column_stack([oof_lgb, oof_et])
test_matrix = np.column_stack([test_preds_lgb, test_preds_et])

meta_model = Ridge(alpha=1.0)
meta_model.fit(oof_matrix, y_full)

stacked_rmse_per_fold = []
for _, val_idx in tscv.split(X_full):
    val_mean = train_data.iloc[val_idx]['tma_mean'].values
    val_std = train_data.iloc[val_idx]['tma_std'].values
    p_norm = meta_model.predict(oof_matrix[val_idx])
    p_abs = (p_norm * val_std) + val_mean
    y_abs = (y_full.iloc[val_idx].values * val_std) + val_mean
    stacked_rmse_per_fold.append(np.sqrt(mean_squared_error(y_abs, p_abs)))

print(f"Ridge Bobot Optimal: LGBM={meta_model.coef_[0]:.4f} | ExtraTrees={meta_model.coef_[1]:.4f}")
print(f"Rata-rata K-Fold Stacked RMSE (denormalized): {np.mean(stacked_rmse_per_fold):.4f}")

Ridge Bobot Optimal: LGBM=1.4503 | ExtraTrees=-0.2915
Rata-rata K-Fold Stacked RMSE (denormalized): 1.7151


## 10. Denormalisasi & Output Final

In [16]:
final_preds_norm = meta_model.predict(test_matrix)

test_mean = test_data['tma_mean'].values
test_std = test_data['tma_std'].values
final_tma = (final_preds_norm * test_std) + test_mean

test_data['tma_mdpl'] = final_tma
submission = test_data[['id', 'tma_mdpl']]

if not os.path.exists('../submissions'):
    os.makedirs('../submissions')

submission.to_csv('../submissions/submission.csv', index=False)
print("Eksperimen 8 selesai. File submission.csv tersimpan.")
print(f"Rentang prediksi TMA: {final_tma.min():.2f} - {final_tma.max():.2f}")

Eksperimen 8 selesai. File submission.csv tersimpan.
Rentang prediksi TMA: 0.73 - 144.43
